# 7B capability check: is the reasoning-arm negative about the model, or the task?

Fifth notebook. Every prior notebook established one thing: Qwen2.5-VL-3B
cannot grade handwritten math above chance (51.7% on a 0.500 baseline,
`03_scaleup_n300.ipynb`), and the simplest prompt fixes do not change that
(`04_confidence_and_prompts.ipynb`). Neither result tells us whether the
problem is this specific 3B model or the task itself.

**This notebook is a gate, not a measurement.** Per the report's own stated
discipline (`report/report.tex`, Recommended Next Steps): check grading
accuracy against the 0.500 baseline *first*, before computing any entropy or
AUROC. Interpreting an uncertainty metric over a grader that cannot do the
task is the exact mistake this project spent its middle phase correcting --
see `pilot.plotting.classify_capability_check`, which encodes the same three
bands the report describes (`at_chance` / `marginal` / `capable`) as a pure,
tested decision function rather than an eyeballed call.

**Reuses the exact same 300 balanced items** as every prior notebook (same
seed, same 50/50 `has_error` split), so a `capable` verdict here can be
compared directly against the 3B numbers with no confound from a different
sample.

**Model:** `Qwen2.5-VL-7B-Instruct` by default. Falls back to a 4-bit
quantized load if the full-precision load does not fit -- flagged loudly,
since a quantized model is not the same measurement as the unquantized one
and that should never be silently glossed over in the results.

**Run order:** cells 1-4 (install, auth, model, sample), then 5 (grading
generation), then 6 (the gate + conditional analysis), then 7 (save).
Transcription is deliberately out of scope -- perception is already settled;
this notebook only touches grading.

In [1]:
# Install cell: GPU-dependent packages only.
# `datasets` is intentionally also in the local requirements.txt -- each
# environment installs its own copy independently, no conflict.
# torch is not installed explicitly: Colab GPU runtimes ship with it preinstalled.
# sentence-transformers is installed here, not in the optional NLI cell at
# the end, so pip resolves every version constraint ONCE before the model
# loads. Installing it mid-session could pull a different transformers
# version underneath an already-loaded model.
!pip install -q transformers accelerate bitsandbytes datasets qwen-vl-utils sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 66.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 74.9 MB/s eta 0:00:00:00:0100:01


In [2]:
# Auth & code/results access cell.
import json
import os
from getpass import getpass

from huggingface_hub import login

# --- Drive mount first: it holds both the model cache and the token store ---
from google.colab import drive

drive.mount("/content/drive")
PROJECT_DIR = "/content/drive/MyDrive/uncertainty-math-vlm"
DRIVE_MODEL_CACHE = f"{PROJECT_DIR}/model_cache"
os.makedirs(DRIVE_MODEL_CACHE, exist_ok=True)

# --- Tokens: entered ONCE, then cached on your Drive ---
# Deliberately not hardcoded in this notebook. This file is tracked in a
# public repo, and GitHub's secret scanning auto-revokes any ghp_ token that
# lands in a public commit -- so an inline token would stop working by
# itself. Drive is private to your account, survives runtime recycling, and
# git never touches it, so you get the same "no retyping" result safely.
TOKEN_FILE = f"{PROJECT_DIR}/.tokens.json"
RESET_TOKENS = False  # set True once to replace previously saved tokens


def get_token(name, prompt):
    """Return a saved token, prompting (once) and persisting it if absent."""
    tokens = {}
    if os.path.exists(TOKEN_FILE):
        with open(TOKEN_FILE) as f:
            tokens = json.load(f)
    if RESET_TOKENS or not tokens.get(name):
        tokens[name] = getpass(prompt).strip()
        with open(TOKEN_FILE, "w") as f:
            json.dump(tokens, f)
        os.chmod(TOKEN_FILE, 0o600)
        print(f"Saved {name} to Drive -- you will not be asked for it again.")
    return tokens[name]


HF_TOKEN = get_token("HF_TOKEN", "Hugging Face token (asked once): ")
GH_TOKEN = get_token("GH_TOKEN", "GitHub token with 'repo' scope (asked once): ")

if not HF_TOKEN.startswith("hf_"):
    raise ValueError(
        "Stored Hugging Face token does not start with 'hf_'. Set "
        "RESET_TOKENS = True and re-run this cell to replace it."
    )

login(token=HF_TOKEN)
print("Hugging Face login OK")

# --- Clone the repo (code + results live in the same repo for this pilot) ---
# Cloned anonymously: the repo is public, so read access needs no token, and
# keeping the token out of the clone URL means a clone error can never echo
# it into this notebook's saved output. The token is used only to push.
REPO_URL = "https://github.com/sepehrmaleki369/uncertainty-math-vlm.git"

# Remove any stale clone from a previous (possibly failed) run so this cell
# is safe to re-run -- git clone silently no-ops into a pre-existing
# directory, which would otherwise leave `repo/` incomplete without error.
!rm -rf repo
!git clone -q {REPO_URL} repo

# %pip (not !pip) installs into the *running kernel's* environment -- !pip
# can silently target a different Python install.
%pip install -q -e repo/

# An editable install writes an `__editable__.pilot-*.pth` file into
# site-packages, but .pth files are only processed by the `site` module at
# INTERPRETER STARTUP. The kernel is already running, so it never sees them
# and `import pilot` fails with ModuleNotFoundError even though the install
# reported success. Putting the repo on sys.path directly makes the package
# importable right now, with no kernel restart needed.
import importlib
import sys

REPO_DIR = os.path.abspath("repo")
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
importlib.invalidate_caches()

import pilot.data
import pilot.prompts
import pilot.parsing
import pilot.entropy

print(f"pilot package imported from: {os.path.dirname(pilot.__file__)}")

Mounted at /content/drive
Hugging Face login OK
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for pilot (pyproject.toml) ... done
pilot package imported from: /content/repo/pilot


In [ ]:
# Model load cell. 7B by default -- this notebook exists specifically to test
# a model bigger than the 3B every other result in this project used.
#
# Falls back to a 4-bit quantized load if full precision does not fit, but
# says so loudly: a quantized model is a genuinely different measurement, not
# a transparent substitute, and silently reporting a quantized-model accuracy
# as if it were the same experiment would misrepresent the capability check.
import torch
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
QUANTIZED = False

try:
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        cache_dir=DRIVE_MODEL_CACHE,
    )
    print(f"Loaded {MODEL_ID} in bfloat16 (full precision).")
except torch.cuda.OutOfMemoryError:
    print(f"bfloat16 load of {MODEL_ID} did not fit -- falling back to 4-bit "
          "quantization. This changes what is being measured; the saved "
          "results record QUANTIZED=True so this is never silently glossed over.")
    from transformers import BitsAndBytesConfig

    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        quantization_config=quantization_config,
        device_map="auto",
        cache_dir=DRIVE_MODEL_CACHE,
    )
    QUANTIZED = True

processor = AutoProcessor.from_pretrained(MODEL_ID, cache_dir=DRIVE_MODEL_CACHE)

if torch.cuda.is_available():
    vram_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU: {torch.cuda.get_device_name(0)} ({vram_gib:.1f} GiB), "
          f"quantized={QUANTIZED}")

In [ ]:
# Sample cell. Must reproduce the reference n=300 run's items exactly, so a
# capability verdict here is directly comparable to the 3B numbers -- same
# function, same seed, same balance as 03/04.
import logging

import pilot.data

logging.basicConfig(level=logging.WARNING, force=True)

N = 300
SEED = 42
TARGET_ERROR_FRAC = 0.5

sample = pilot.data.load_fermat_balanced(
    n=N, seed=SEED, target_error_frac=TARGET_ERROR_FRAC
)
N = len(sample)
n_error = sum(bool(x) for x in sample["has_error"])
print(f"{N} items, {n_error} with a mistake, {N - n_error} clean "
      f"({n_error / N:.0%} error rate)")

In [ ]:
# Grading-only generation. No transcription (perception is already settled),
# no log-probability capture (no vision-tower slowdown risk -- see 04's OOM
# history; output_scores is simply never requested here). K=5, matching every
# prior grading run for direct comparability.
#
# Batched with the same adaptive backoff ladder that made notebook 04 robust:
# start at the full batch, back off on OOM, and STAY backed off rather than
# retrying a failing size on every item.
import gc
import json
import os
import time

import torch
from qwen_vl_utils import process_vision_info
from tqdm.auto import tqdm

import pilot.prompts

K_GRADING = 5
TEMP = 0.7
_BATCH_LADDER = [5, 2, 1]
_batch_state = {"index": 0}

META_FIELDS = ("orig_q", "pert_a", "has_error", "handwriting_style", "image_quality")
INFRA_EXCEPTIONS = (ConnectionError, TimeoutError, torch.cuda.OutOfMemoryError, OSError)


def _generate_batch(messages, n: int, temperature: float):
    text_prompt = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text_prompt], images=image_inputs, videos=video_inputs,
        padding=True, return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs, max_new_tokens=512, do_sample=True,
            temperature=temperature, num_return_sequences=n,
        )

    trimmed = output_ids[:, inputs["input_ids"].shape[1]:]
    texts = processor.batch_decode(
        trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )
    del output_ids, inputs
    gc.collect()
    torch.cuda.empty_cache()
    return texts


def generate_grading(messages, n: int, temperature: float):
    """Draw n grading samples, backing the batch size off on OOM and staying
    there -- 7B is a larger model than any prior notebook used, so even
    without logprob capture the batch that fit for 3B may not fit here."""
    texts = []
    last_exc = None
    while len(texts) < n:
        want = n - len(texts)
        size = min(_BATCH_LADDER[_batch_state["index"]], want)
        for attempt in range(3):
            try:
                texts += _generate_batch(messages, size, temperature)
                last_exc = None
                break
            except torch.cuda.OutOfMemoryError:
                gc.collect()
                torch.cuda.empty_cache()
                if _batch_state["index"] + 1 < len(_BATCH_LADDER):
                    _batch_state["index"] += 1
                    print(f"  OOM at batch {size}; dropping to "
                          f"{_BATCH_LADDER[_batch_state['index']]} for the rest of the run.",
                          flush=True)
                    size = min(_BATCH_LADDER[_batch_state["index"]], n - len(texts))
                    continue
                raise
            except INFRA_EXCEPTIONS as exc:
                last_exc = exc
                gc.collect()
                torch.cuda.empty_cache()
                if attempt < 2:
                    time.sleep(5)
        if last_exc is not None:
            raise last_exc
    return texts


CHECKPOINT_DIR = "/content/drive/MyDrive/uncertainty-math-vlm/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
model_slug = MODEL_ID.split("/")[-1]
# Quantization mode is part of the checkpoint key for the same reason
# CAPTURE_LOGPROBS was in notebook 04: a skip or a sample drawn under one
# mode is not evidence about the other, and reusing the filename across modes
# would silently mix results from two different measurements.
grading_path = (f"{CHECKPOINT_DIR}/grading_7b_k{K_GRADING}_{model_slug}"
                f"_n{N}_seed{SEED}{'_4bit' if QUANTIZED else ''}.jsonl")

grading_results = []
if os.path.exists(grading_path):
    with open(grading_path) as f:
        grading_results = [json.loads(line) for line in f if line.strip()]
    valid = []
    for idx, entry in enumerate(grading_results[:N]):
        item = sample[idx]
        if not all(entry["item"].get(k) == item[k] for k in META_FIELDS):
            print(f"Checkpoint item {idx + 1} does not match sample order; resuming there.")
            break
        if len(entry.get("samples_raw", [])) != K_GRADING:
            break
        valid.append(entry)
    if len(valid) != len(grading_results):
        with open(grading_path, "w") as f:
            for e in valid:
                f.write(json.dumps(e, default=str) + "\n")
    grading_results = valid
    print(f"Resuming from {len(grading_results)} completed items")

if len(grading_results) >= N:
    print(f"All {N} items already done.")
else:
    with tqdm(total=(N - len(grading_results)) * K_GRADING, desc="grading", unit="sample") as pbar:
        for item_idx, item in enumerate(sample):
            if item_idx < len(grading_results):
                continue
            messages = pilot.prompts.build_grading_messages(item["image"])
            texts = generate_grading(messages, K_GRADING, TEMP)
            entry = {
                "item": {k: item[k] for k in META_FIELDS},
                "samples_raw": texts,
                "quantized": QUANTIZED,
            }
            grading_results.append(entry)
            with open(grading_path, "a") as f:
                f.write(json.dumps(entry, default=str) + "\n")
                f.flush()
            pbar.update(K_GRADING)

print(f"grading_results: {len(grading_results)} items")

In [ ]:
# The gate. Accuracy first, entropy second -- never the other way around.
import importlib

import numpy as np
import pandas as pd

import pilot.entropy
import pilot.parsing
import pilot.plotting

for m in (pilot.parsing, pilot.entropy, pilot.plotting):
    importlib.reload(m)

rows = []
for entry in grading_results:
    digits = [pilot.parsing.parse_grading(t) for t in entry["samples_raw"]]
    labels = [None if d is None else str(d) for d in digits]
    majority, _ = pilot.entropy.majority_cluster(labels)
    said_error = majority == "1"
    rows.append({
        "orig_q": entry["item"]["orig_q"],
        "pert_a": entry["item"]["pert_a"],
        "has_error": entry["item"]["has_error"],
        "grading_correct": majority in {"0", "1"} and said_error == bool(entry["item"]["has_error"]),
        "said_error": said_error,
        "reasoning_entropy": pilot.entropy.cluster_entropy(labels),
        "n_grading_parse_failures": sum(1 for d in digits if d is None),
        "model_id": MODEL_ID,
        "quantized": QUANTIZED,
    })
df = pd.DataFrame(rows)

baseline = pilot.plotting.majority_class_baseline(df, "has_error")
accuracy = float(df["grading_correct"].mean())
gate = pilot.plotting.classify_capability_check(
    accuracy=accuracy, baseline_accuracy=baseline["baseline_accuracy"], n_items=len(df),
)

print("=" * 70)
print("CAPABILITY GATE")
print("=" * 70)
print(f"  model               {MODEL_ID}{' (4-bit quantized)' if QUANTIZED else ''}")
print(f"  grading accuracy    {gate['accuracy']:.3f}")
print(f"  baseline accuracy   {gate['baseline_accuracy']:.3f}  (majority-class)")
print(f"  margin              {gate['margin_over_baseline']:+.3f}")
print(f"  VERDICT             {gate['verdict'].upper()}")
print()

if gate["verdict"] == "at_chance":
    print("Grades at or near chance. The 3B negative result generalizes to this")
    print("model too: entropy over a grader that cannot do the task has nothing")
    print("to measure. The AUROC below is computed for the record only and")
    print("should NOT be read as evidence about reasoning entropy either way.")
elif gate["verdict"] == "marginal":
    print("Some signal above baseline, but below the report's 0.65 threshold for")
    print("treating the entropy question as live again. The AUROC below is")
    print("informative but should be read as suggestive, not conclusive.")
else:
    print("Clears the capability bar. The AUROC below is a genuine test of")
    print("reasoning entropy -- unlike every prior grading result in this")
    print("project, this model can plausibly do the task.")

print()
print("=" * 70)
print(f"REASONING ENTROPY AUROC ({'meaningful' if gate['entropy_result_meaningful'] else 'diagnostic only'})")
print("=" * 70)
r = pilot.plotting.bootstrap_auroc_ci(df, "reasoning_entropy", "grading_correct", n_boot=10000, seed=0)
print(f"  pooled              AUROC {r['auroc']:.3f} [{r['ci_low']:.3f}, {r['ci_high']:.3f}]  "
      f"n_wrong={r['n_error']}")

gt = df["has_error"].astype(bool)
for mask, name in ((gt, "has_error=1 stratum"), (~gt, "clean stratum")):
    sub = df[mask]
    if sub["grading_correct"].nunique() < 2:
        print(f"  {name:20s} single class, AUROC undefined")
        continue
    rs = pilot.plotting.bootstrap_auroc_ci(sub, "reasoning_entropy", "grading_correct", n_boot=10000, seed=0)
    print(f"  {name:20s} AUROC {rs['auroc']:.3f} [{rs['ci_low']:.3f}, {rs['ci_high']:.3f}]  "
          f"n_wrong={rs['n_error']}")

print()
print(f"Parse failures: {int(df['n_grading_parse_failures'].sum())}/{len(df) * K_GRADING} "
      f"({df['n_grading_parse_failures'].sum() / (len(df) * K_GRADING):.1%})")
print(f"Answers \"there is a mistake\": {df['said_error'].mean():.0%} of items "
      f"(3B was 93%)")

In [ ]:
# Save cell: Drive first, then repo + push.
import subprocess
from datetime import datetime, timezone
from getpass import getpass

timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
csv_name = (f"grading_7b_n{N}_{'4bit_' if QUANTIZED else ''}"
            f"{model_slug.lower()}_{timestamp}.csv")

drive_results = "/content/drive/MyDrive/uncertainty-math-vlm/results"
os.makedirs(drive_results, exist_ok=True)
df.to_csv(f"{drive_results}/{csv_name}", index=False)
print(f"Backup written to {drive_results}/{csv_name}")

os.makedirs("repo/results", exist_ok=True)
csv_path = f"repo/results/{csv_name}"
df.to_csv(csv_path, index=False)
print(f"Wrote {csv_path} ({len(df)} rows)")

_REDACT = []


def git(*args):
    result = subprocess.run(["git", "-C", "repo", *args], capture_output=True, text=True)
    output = (result.stdout or "") + (result.stderr or "")
    for secret in _REDACT:
        if secret:
            output = output.replace(secret, "***")
    if result.returncode != 0 and output.strip():
        print(output.strip())
    return result


git("config", "user.email", "colab-pilot@localhost")
git("config", "user.name", "Colab Pilot Run")
git("add", f"results/{csv_name}")
commit = git("commit", "-m", f"Add 7B grading capability check: {csv_name}")
if commit.returncode != 0:
    print("git commit failed (see above) -- CSV is safe on Drive.")

GH_PUSH_TOKEN = (globals().get("GH_TOKEN") or "").strip()
if not GH_PUSH_TOKEN:
    GH_PUSH_TOKEN = getpass("GitHub token (to push results), then press Enter: ").strip()
_REDACT.append(GH_PUSH_TOKEN)

if not GH_PUSH_TOKEN:
    print("No token given -- skipping push. CSV is saved on Drive and in repo/results/.")
else:
    push_url = REPO_URL.replace("https://", f"https://{GH_PUSH_TOKEN}@")
    if git("fetch", push_url, "main").returncode == 0:
        if git("rebase", "FETCH_HEAD").returncode != 0:
            git("rebase", "--abort")
            print("Rebase onto remote failed; attempting push anyway.")
    if git("push", push_url, "HEAD:main").returncode == 0:
        print("Pushed results.")
    else:
        print("Push failed (see above). The CSV is safe on Drive and in "
              "repo/results/ -- retry the push without re-running the model.")